In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/05_dim_payer
# Purpose  : SCD Type 1 via dlt.apply_changes.
#            Financial metrics (REVENUE, COVERED_ENCOUNTERS, etc.) were intentionally
#            omitted here—they are measures, not descriptive attributes, and belong in
#            fact_payer_summary if required later (Kimball methodology: avoid bloat in Dimension tables).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_payer_changes")
def v_payer_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.payers")
    return df.select(
        F.xxhash64("payer_id").alias("payer_key"),
        "payer_id",
        "payer_name",
        "address",
        "city",
        "state_headquartered",
        "zip",
        "phone",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_payer")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_payer",
    source="v_payer_changes",
    keys=["payer_id"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)